# Week 11: Text classification and embeddings (capstone milestone M3)

| Session | Topic | Time |
|---|---|---|
| Day 1 | Naive Bayes from scratch (and why Laplace smoothing is MAP) | 2.5 h |
| Day 2 | TF-IDF + logistic regression: a strong baseline, error analysis | 2.5 h |
| Day 3 | Word embeddings from co-occurrence + SVD | 2.5 h |
| Day 4 | Dense retrieval with a pretrained sentence encoder | 2 h |
| Review | Spaced recall | 0.5 h |

**Task for Days 1–2:** **PubMed 20k RCT**, sentences from abstracts of randomised controlled trials, each labelled BACKGROUND, OBJECTIVE, METHODS, RESULTS or CONCLUSIONS. Systems like this help researchers skim the literature. (Dernoncourt and Lee, 2017. Check the licence in the GitHub repository and add it to `DATASETS.md`.)

**Read:** *Speech and Language Processing* (3rd edition draft), the chapters on Naive Bayes and on vector semantics and embeddings.

### Done when
- [ ] Your Naive Bayes gives the same predictions as scikit-learn's.
- [ ] The TF-IDF + logistic regression macro-F1 is reported on **dev**, with a confusion matrix and 5 analysed errors.
- [ ] **M3:** a dense retrieval run on NFCorpus, evaluated with **your** metrics code and saved to `capstone/results/`.

**One-time install** (Day 4): `pip install sentence-transformers` (pulls in PyTorch, about 1 GB). On WSL, install the CPU build first to save space: `pip install torch --index-url https://download.pytorch.org/whl/cpu`.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from course.data import load_pubmed_rct
tr_texts, tr_labels, tr_ids = load_pubmed_rct("train", max_sentences=40000)
dev_texts, dev_labels, _ = load_pubmed_rct("dev", max_sentences=10000)
print(len(tr_texts), "train sentences |", tr_labels[0], "→", tr_texts[0][:100])
import collections; print(collections.Counter(tr_labels))

✍️ *The classes are imbalanced. Which metric is better here, accuracy or macro-F1, and why? Note that the official splits are by abstract: why does that matter (Week 8)?* (double-click to edit)



---
# Day 1: Multinomial Naive Bayes from scratch (2.5 h)

**The model:** pick the class $c$ that maximises $P(c \mid \text{doc}) \propto P(c) \prod_{w} P(w \mid c)^{\text{count}(w)}$, and do it in **log space** (Week 7!):
$$\hat c = \arg\max_c \; \log P(c) + \sum_w \text{count}(w)\,\log \theta_{c,w}$$
**Training is just counting** (it's the MLE): $P(c)$ = the share of documents in class $c$; $\theta_{c,w} = \frac{N_{c,w}}{N_c}$.

**Problem:** a word never seen in class $c$ gives $\log 0 = -\infty$. **Fix:** Laplace smoothing, $\theta_{c,w} = \frac{N_{c,w} + \alpha}{N_c + \alpha |V|}$. That's exactly **MAP with a Dirichlet prior**: the same pattern as the coin with a Beta prior. "Naive" refers to assuming the words are independent given the class. That's false, yet it works surprisingly well.

In matrix form, with a count matrix $X$ (docs × vocab): `scores = X @ log_theta.T + log_prior`. One matrix multiplication classifies everything.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer(min_df=2)
X_tr = vec.fit_transform(tr_texts)            # sparse matrix: most entries are 0
X_dev = vec.transform(dev_texts)              # transform only: the vocabulary comes from train
y_tr, y_dev = np.array(tr_labels), np.array(dev_labels)
print(X_tr.shape, f"{X_tr.nnz / np.prod(X_tr.shape):.4%} non-zero")

### Exercise 11.1: `NaiveBayes`
Implement `fit(X, y)` and `predict(X)`, with `alpha=1.0`. Store `self.classes_` (sorted unique labels), `self.log_prior_` (shape C), and `self.log_theta_` (shape C × V). `X` is a scipy sparse matrix: `X[mask].sum(axis=0)` works, and gives a 1×V matrix. Use `np.asarray(...).ravel()` to turn it into a vector.

In [ ]:
class NaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        raise NotImplementedError

    def predict(self, X):
        raise NotImplementedError

In [ ]:
from sklearn.naive_bayes import MultinomialNB
ours = NaiveBayes().fit(X_tr, y_tr).predict(X_dev)
ref = MultinomialNB(alpha=1.0).fit(X_tr, y_tr).predict(X_dev)
print("agreement with sklearn:", (ours == ref).mean(), "| dev accuracy:", (ours == y_dev).mean().round(4))
assert (ours == ref).mean() > 0.999
print("✅ 11.1 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`fit` must `return self`, so that `.fit(...).predict(...)` works.

</details>

<details><summary>Hint 2</summary>

For each class c: `counts = np.asarray(X[y == c].sum(axis=0)).ravel()`, then `log_theta[c] = np.log((counts + alpha) / (counts.sum() + alpha * V))`.

</details>

<details><summary>Hint 3</summary>

`predict`: `scores = X @ self.log_theta_.T + self.log_prior_`, then `self.classes_[scores.argmax(axis=1)]`. Wrap with `np.asarray` if you get a matrix type.

</details>


---
# Day 2: TF-IDF + logistic regression, and reading your errors (2.5 h)

### Exercise 11.2: A strong baseline
Build `clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), LogisticRegression(max_iter=2000, C=…))`. Choose `C` from {0.3, 1, 3, 10} by **dev macro-F1** (Week 7: `C` is the *inverse* of the L2 strength). Store the best fitted pipeline in `clf` and its dev macro-F1 in `macro_f1`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

clf, macro_f1 = None, None

In [ ]:
nb_f1 = f1_score(y_dev, ours, average="macro")
print(f"NaiveBayes macro-F1 {nb_f1:.3f} | TF-IDF+LR macro-F1 {macro_f1:.3f}")
assert macro_f1 > nb_f1 - 0.02
pred = clf.predict(dev_texts)
print(classification_report(y_dev, pred, digits=3))
ConfusionMatrixDisplay.from_predictions(y_dev, pred, xticks_rotation=45, normalize="true", values_format=".2f"); plt.show()
print("✅ 11.2 passed")

### Exercise 11.3: What did the model learn?
Write `top_features(clf, label, n=10)`, which returns the `n` n-grams with the largest positive weight for `label`. Use `clf[0].get_feature_names_out()` and `clf[-1].coef_` (one row per class, in the order of `clf[-1].classes_`).

In [ ]:
def top_features(clf, label, n=10):
    raise NotImplementedError

In [ ]:
for lab in clf[-1].classes_:
    feats = top_features(clf, lab)
    assert len(feats) == 10
    print(f"{lab:12s}", ", ".join(feats))
print("✅ 11.3 passed")

✍️ *Which two classes get confused most, and why is that linguistically plausible? Paste 5 misclassified dev sentences and say, for each one, whether the model or the label seems wrong. What context is missing? (Hint: a sentence's *position* in the abstract would help a lot. That's an idea for improvement.)* (double-click to edit)



---
# Day 3: Word embeddings from co-occurrence (2.5 h)

**The distributional hypothesis:** *"You shall know a word by the company it keeps."* Words used in similar contexts get similar vectors.

**The recipe** (the count-based ancestor of word2vec):
1. Count how often word *j* appears within ±`window` positions of word *i*, giving the matrix $C$ (V × V).
2. Re-weight with **PPMI**: $\max\left(0, \log \frac{P(i, j)}{P(i)P(j)}\right)$, so frequent words don't dominate (the same idea as IDF).
3. Take a **truncated SVD** (Week 6) and keep 100 dimensions. Each row is a word vector.

word2vec learns similar vectors by gradient descent, and was later shown to be implicitly factorising a shifted PMI matrix (Levy and Goldberg, 2014).

### Exercise 11.4: `cooccurrence_matrix`
`token_lists` is a list of token lists; `vocab` maps word → index for the top-V words (ignore other words). Return a dense `np.float64` array C, where `C[i, j]` counts the times word j occurs within `window` positions of word i (excluding the position itself). It must be symmetric.

In [ ]:
def cooccurrence_matrix(token_lists, vocab, window=4):
    raise NotImplementedError

In [ ]:
toy = [["a", "b", "c"], ["a", "c"]]
C = cooccurrence_matrix(toy, {"a": 0, "b": 1, "c": 2}, window=1)
assert np.array_equal(C, np.array([[0, 1, 1], [1, 0, 1], [1, 1, 0]]))
assert np.array_equal(cooccurrence_matrix(toy, {"a": 0, "c": 1}, window=2), np.array([[0, 2], [2, 0]]))

from collections import Counter
token_lists = [t.lower().split() for t in tr_texts]
top = [w for w, _ in Counter(w for tl in token_lists for w in tl).most_common(3000)]
vocab = {w: i for i, w in enumerate(top)}
C = cooccurrence_matrix(token_lists, vocab, window=4)
total = C.sum(); pi = C.sum(axis=1, keepdims=True) / total
with np.errstate(divide="ignore", invalid="ignore"):
    ppmi = np.maximum(np.log((C / total) / (pi @ pi.T)), 0)
ppmi = np.nan_to_num(ppmi)
U, s, _ = np.linalg.svd(ppmi, full_matrices=False)
E = U[:, :100] * s[:100]
E /= np.linalg.norm(E, axis=1, keepdims=True) + 1e-12

def nearest(word, n=6):
    sims = E @ E[vocab[word]]
    return [top[i] for i in np.argsort(-sims)[1:n + 1]]
for w in [w for w in ["placebo", "randomized", "significantly", "patients", "aim"] if w in vocab]:
    print(f"{w:14s} →", nearest(w))
print("✅ 11.4 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Loop over each token list. For position i (if its token is in vocab), loop over j in `range(max(0, i - window), min(len(tl), i + window + 1))`, skip j == i, and if `tl[j]` is in vocab, add 1 to `C[vocab[tl[i]], vocab[tl[j]]]`.

</details>

<details><summary>Hint 2</summary>

Counting both (i, j) and (j, i) during the loop makes it symmetric automatically.

</details>


✍️ *Are the neighbours sensible? Find one surprising neighbour and explain it from the contexts the word appears in.* (double-click to edit)



---
# Day 4: Dense retrieval (capstone milestone M3) (2 h)

A **sentence encoder** (here `all-MiniLM-L6-v2`, about 22M parameters, Apache-2.0 licence) maps whole texts to 384-dimensional unit vectors, so that texts with similar *meaning* end up close together, even when they share no words. Retrieval becomes: encode the corpus once, encode the query, take the top-k by cosine (Week 5's `cosine_matrix`!).

`course.embeddings.encode` caches the corpus embeddings in `data/embeddings/`. Encoding 3,600 NFCorpus documents takes a few minutes on your CPU, once.

In [ ]:
from course.data import load_nfcorpus
from course.embeddings import encode
metrics = fresh("medlab.metrics")
corpus, queries, qrels = load_nfcorpus("test")
doc_ids = list(corpus)
doc_emb = encode([corpus[d]["title"] + " " + corpus[d]["text"] for d in doc_ids], cache_name="nfcorpus_minilm")
q_ids = list(queries)
q_emb = encode([queries[q] for q in q_ids])
print(doc_emb.shape, q_emb.shape)

### Exercise 11.5: `capstone/medlab/dense.py`
```python
def dense_search(doc_emb, doc_ids, q_emb, q_ids, k=100) -> dict[str, dict[str, float]]   # a run
```
The embeddings are already unit-normalised, so cosine = dot product. Score all queries against all documents in **one** matrix multiplication. For the top k, use `np.argpartition` (O(n)) followed by sorting only those k (much faster than a full sort over a large corpus). Add a test in `tests/test_dense.py` with tiny hand-made vectors.

In [ ]:
# Implement medlab/dense.py and a test, then run the checker.

In [ ]:
import pandas as pd
dense = fresh("medlab.dense")
assert run_tests("tests/test_dense.py") == 0
run_dense = dense.dense_search(doc_emb, doc_ids, q_emb, q_ids, k=100)
assert set(run_dense) == set(q_ids) and all(len(v) == 100 for v in run_dense.values())
first = run_dense[q_ids[0]]
best_doc = doc_ids[int(np.argmax(doc_emb @ q_emb[0]))]
assert max(first, key=first.get) == best_doc
res = metrics.evaluate_run(run_dense, qrels)
print({k: round(v, 4) for k, v in res.items()})
Path("../capstone/results").mkdir(parents=True, exist_ok=True)
import json; json.dump(run_dense, open("../capstone/results/run_dense_test.json", "w"))
lex = pd.read_csv("../capstone/results/week10_lexical.csv", index_col=0)
print(pd.concat([lex, pd.DataFrame([res], index=["Dense (MiniLM)"]).round(4)]))
print("✅ 11.5 passed (milestone M3)")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`S = q_emb @ doc_emb.T` gives shape (n_queries, n_docs).

</details>

<details><summary>Hint 2</summary>

For row i: `top = np.argpartition(-S[i], k)[:k]`, then sort `top` by `-S[i, top]`. Build `{doc_ids[j]: float(S[i, j]) for j in top}`. Guard against `k >= n_docs`.

</details>


✍️ *Does dense beat BM25 on NFCorpus? (In the BEIR paper, many dense models *don't*: domain shift.) Find one query where dense wins and one where BM25 wins, and explain both.* (double-click to edit)



---
# Review (30 min)
1. *(Week 6)* What does a truncated SVD keep, and what does it throw away?
2. *(Week 7)* Laplace smoothing is MAP with which prior?
3. *(Week 8)* What's the difference between macro-F1 and accuracy?
4. *(Week 10)* Write the BM25 TF component from memory.
5. Why must the `CountVectorizer` be fitted on train only?
6. What is PPMI, and how is it similar to IDF?
7. Name one strength of dense retrieval and one of BM25.

✍️ *Your answer:* (double-click to edit)

